# V19 – Datenbanken Teil 1: Python-Recap

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- den `with`-Kontext-Manager sicher für Dateien und Datenbank-Verbindungen verwenden,
- ein **Tupel** mit mehreren Variablen in einer Zeile entpacken,
- eine Liste von Tupeln sauber in einer `for`-Schleife durchlaufen,
- eine CSV-Datei mit `csv.DictReader` zeilenweise einlesen,
- Werte über **f-Strings** lesbar formatieren.

## ⏱️ Zeitbudget
Ca. 15 Minuten als Warm-up vor der eigentlichen Theorie.

## 🧭 TL;DR
Das `sqlite3`-Modul liefert dir Daten fast immer als **Liste von Tupeln** zurück. Damit du diese Daten anschließend ohne hässliche Index-Zugriffe (`row[0]`, `row[1]`, …) verarbeitest, frischen wir Tupel-Entpacken, `for`-Schleifen und f-Strings auf. Außerdem nutzen wir gleich in der Praxis den `with`-Block, damit Verbindungen und Dateien zuverlässig geschlossen werden.

## 📦 Voraussetzungen
- Python-Grundlagen aus V01–V10.
- Du kannst Zellen mit `Shift+Enter` ausführen.


## 1. Der `with`-Kontext-Manager

Ein **Kontext-Manager** sorgt dafür, dass eine begonnene Ressource am Ende eines Blocks zuverlässig wieder aufgeräumt wird – auch wenn mittendrin ein Fehler auftritt. Den typischen Anwendungsfall kennst du aus Dateien, bei denen `with open(...)` das spätere `close()` überflüssig macht.

> [!NOTE]
> Ein **Kontext-Manager** ist ein Objekt, das über `__enter__` und `__exit__` die Lebensdauer einer Ressource klammert. Der `with`-Block garantiert Aufräum-Arbeiten, selbst wenn im Block eine Exception fliegt.

Genau dieselbe Idee nutzt gleich `sqlite3.connect(...)`: Nach dem Block ist die Datenbank-Verbindung automatisch committed (bei erfolgreichem Durchlauf) oder zurückgerollt (bei einer Exception).


In [1]:
# Datei lesen - klassisch mit with-Block
from pathlib import Path
demo = Path("_demo.txt")
demo.write_text("Zeile 1\nZeile 2\n", encoding="utf-8")

with open(demo, encoding="utf-8") as f:
    inhalt = f.read()
print(inhalt)

demo.unlink()  # Aufraeumen


Zeile 1
Zeile 2



## 2. Tupel und Tupel-Entpacken

Ein **Tupel** ist eine unveränderliche, geordnete Sammlung von Werten, geschrieben in runden Klammern: `(1, "Zahnrad", 25, 1.8)`. `sqlite3` liefert jede Datenbank-Zeile genau in diesem Format zurück.

> [!NOTE]
> Beim **Tupel-Entpacken** weist Python die einzelnen Komponenten eines Tupels in einem Schritt mehreren Variablen zu. Die Anzahl der Variablen auf der linken Seite muss mit der Anzahl der Elemente übereinstimmen.


In [2]:
zeile = (1, "Zahnrad Z42", 25, 1.8)

# Variante A: Zugriff ueber Index (unleserlich)
print("Name (Index):", zeile[1])

# Variante B: Tupel-Entpacken (klar benannte Variablen)
produkt_id, name, zeit, masse = zeile
print(f"Name: {name}, Zeit: {zeit} min, Masse: {masse} kg")


Name (Index): Zahnrad Z42
Name: Zahnrad Z42, Zeit: 25 min, Masse: 1.8 kg


## 3. Liste von Tupeln in einer Schleife

Das Ergebnis von `cursor.fetchall()` ist fast immer eine **Liste von Tupeln**. In der `for`-Schleife kannst du jedes Tupel direkt wieder entpacken, ohne Zwischenvariable.


In [3]:
zeilen = [
    (1, "Zahnrad Z42", 25, 1.8),
    (2, "Welle W-18", 35, 3.2),
    (3, "Flansch F-90", 18, 2.5),
]

for produkt_id, name, zeit, masse in zeilen:
    print(f"{produkt_id:>2} | {name:<16} | {zeit:>3} min | {masse:.1f} kg")


 1 | Zahnrad Z42      |  25 min | 1.8 kg
 2 | Welle W-18       |  35 min | 3.2 kg
 3 | Flansch F-90     |  18 min | 2.5 kg


> [!TIP]
> Wenn du nur ein einzelnes Feld brauchst, kannst du ungenutzte Variablen mit einem Unterstrich markieren, zum Beispiel `for _, name, _, _ in zeilen:`. Python wertet das als „mich interessiert dieses Feld nicht".


## 4. `csv.DictReader` – CSV zeilenweise als Dictionary

Die Klasse `csv.DictReader` liest eine CSV-Datei und liefert pro Zeile ein **Dictionary**, dessen Schlüssel den Spaltennamen aus der Header-Zeile entsprechen. Das macht den Code unabhängig von der Spalten-Reihenfolge.

> [!NOTE]
> Ein **Dictionary** ist eine Zuordnung von **Schlüsseln** zu **Werten**. Mit `zeile["Produktname"]` greifst du auf den Wert zu, den die aktuelle Zeile in der Spalte `Produktname` hat.


In [4]:
import csv

with open("testdaten/produkte.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for zeile in reader:
        # CSV liefert ALLES als String - Typen bei Bedarf konvertieren
        name = zeile["Produktname"]
        zeit = int(zeile["Produktionszeit_Minuten"])
        print(f"{name:<25} -> {zeit} min")


Zahnrad Z42               -> 25 min
Welle W-18                -> 35 min
Flansch F-90              -> 18 min
Buchse B-25               -> 12 min
Gehaeuse G-55             -> 45 min
Verbindungselement VE-12  -> 8 min


> [!WARNING]
> `csv.DictReader` liefert alle Felder als **Strings**. Willst du mit Zahlen rechnen oder sie in eine `INTEGER`- oder `REAL`-Spalte einer Datenbank schreiben, musst du vorher mit `int(...)` bzw. `float(...)` konvertieren. Sonst landet die Zahl als Text in der DB.


## 5. f-Strings kompakt

**f-Strings** (formatierte String-Literale) setzen Werte direkt in einen Text ein. Innerhalb der geschweiften Klammern stehen Ausdrücke, optional gefolgt von einem Format-Spezifizierer nach einem Doppelpunkt.


In [5]:
name = "Welle W-18"
zeit = 35
masse = 3.2

print(f"{name} benoetigt {zeit} Minuten und {masse:.2f} kg Material.")
print(f"Links: '{name:<15}' | Rechts: '{zeit:>5}'")


Welle W-18 benoetigt 35 Minuten und 3.20 kg Material.
Links: 'Welle W-18     ' | Rechts: '   35'


## 6. Mini-Übung 1 — Tupel entpacken

Die Liste `messungen` enthält drei Tupel mit `(sensor_id, wert)`. Summiere alle `wert`-Einträge in die Variable `gesamt`.


In [6]:
messungen = [("T01", 22.5), ("T02", 24.1), ("T03", 21.7)]
gesamt = 0.0  # TODO: alle Werte aufsummieren


In [7]:
# Selbstkontrolle
try:
    assert abs(gesamt - 68.3) < 1e-6, f"Erwartet 68.3, bekommen: {gesamt}"
    print("✅ Tupel-Entpacken sitzt!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `gesamt` existiert noch nicht.")


❌ Noch nicht ganz: Erwartet 68.3, bekommen: 0.0


## 7. Mini-Übung 2 — CSV lesen und zählen

Zähle mit `csv.DictReader`, wie viele Zeilen die Datei `testdaten/produkte.csv` enthält (ohne Header). Speichere die Anzahl in `anzahl_zeilen`.


In [8]:
import csv

anzahl_zeilen = 0
# TODO: Datei oeffnen, DictReader anlegen, Zeilen zaehlen


In [9]:
# Selbstkontrolle
try:
    assert anzahl_zeilen == 6, f"Erwartet 6 Zeilen, gezaehlt: {anzahl_zeilen}"
    print("✅ CSV-Zeilen korrekt gezaehlt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `anzahl_zeilen` existiert noch nicht.")


❌ Noch nicht ganz: Erwartet 6 Zeilen, gezaehlt: 0


## ✅ Zusammenfassung

- Der `with`-Block übernimmt das Aufräumen von Dateien und DB-Verbindungen.
- Tupel-Entpacken macht `for`-Schleifen über Datenbank-Ergebnisse lesbar.
- `csv.DictReader` liest CSV-Zeilen als Dictionary, alle Werte sind zunächst Strings.
- f-Strings sind der Standardweg, Werte in Ausgabe-Texte einzubetten.

## ➡️ Nächster Schritt
Weiter mit [01_theorie.ipynb](01_theorie.ipynb) — dort lernst du das relationale Modell und SQL kennen.
